# Practical 01: Natural Language Preprocessing Pipeline (NLTK vs spaCy)

**Department of Computer Engineering, Sanjivani College of Engineering, Kopargaon**  
**Course:** Natural Language Processing (BTech Computer Engineering SEM - VII)  
**Course Outcome:** CO1 — Demonstrate core NLP preprocessing pipelines, tokenization, stemming, lemmatization, and POS tagging.

---

### Objectives:
1. Implement a complete text preprocessing pipeline using **Python (NLTK and spaCy)**.
2. Explore different **Tokenization** techniques: Whitespace, Rule-based (Treebank, WordPunct, Regexp), and Subword tokenization.
3. Compare **Stop-word removal** strategies (NLTK vs spaCy vs Domain-specific).
4. Contrast **Stemming** algorithms (**PorterStemmer** vs **SnowballStemmer**) and observe over-stemming / under-stemming phenomena.
5. Contrast **Lemmatization** (**WordNetLemmatizer** with POS tags vs **spaCy Lemmatizer**).
6. Implement and benchmark **Part-of-Speech (POS) Tagging** (NLTK Penn Treebank vs spaCy Universal POS).
7. Perform a systematic **side-by-side comparison** of NLTK vs spaCy on raw text corpora.
8. **Real-World Exemplar:** Build a production-grade, reusable preprocessing pipeline to normalize noisy, code-mixed, emoji-laden social media posts scraped from Twitter/Reddit for a **Fake-News Detection System**.


## 1. Setup & Environment Initialization
First, let us import the required libraries and download necessary lexical databases.


In [1]:
# Import standard NLP and Data Processing libraries
import re
import emoji
import pandas as pd
import nltk
import spacy

# Download essential NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

# Load spaCy English pipeline (with automatic download fallback)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("Environment successfully initialized!")
print(f"spaCy Version: {spacy.__version__}")
print(f"NLTK Version:  {nltk.__version__}")


Environment successfully initialized!
spaCy Version: 3.8.16
NLTK Version:  3.9.1


## 2. Tokenization Techniques

Tokenization is the process of breaking a continuous stream of text into smaller discrete units called **tokens** (words, punctuation, subwords).

We compare three paradigms:
1. **Whitespace Tokenization**: Naive split on space. Fails on punctuation attached to words.
2. **Rule-Based Tokenization**: Uses linguistic grammars and regular expressions (e.g., NLTK `word_tokenize`, `WordPunctTokenizer`, `RegexpTokenizer`).
3. **Subword Tokenization**: Breaks words down into morphological components or Byte-Pair subwords to handle Out-Of-Vocabulary (OOV) tokens.


In [2]:
from nltk.tokenize import word_tokenize, WordPunctTokenizer, RegexpTokenizer

sample_text = "Dr. Smith isn't attending NLP-2026 conference! He paid $450.50 at https://nlpconf.org."

# 1. Whitespace Tokenization
whitespace_tokens = sample_text.split()

# 2. NLTK Treebank Tokenizer (handles contractions: isn't -> is, n't)
treebank_tokens = word_tokenize(sample_text)

# 3. WordPunct Tokenizer (splits all punctuation into distinct tokens)
punct_tokenizer = WordPunctTokenizer()
punct_tokens = punct_tokenizer.tokenize(sample_text)

# 4. Custom Regexp Tokenizer (keeps only words and monetary amounts)
custom_regex_tokenizer = RegexpTokenizer(r'\$\d+(?:\.\d+)?|\w+')
regex_tokens = custom_regex_tokenizer.tokenize(sample_text)

print("Original Text:", sample_text)
print("-" * 75)
print("1. Whitespace Split Tokens :", whitespace_tokens[:8])
print("2. Treebank (word_tokenize):", treebank_tokens[:8])
print("3. WordPunct Tokens        :", punct_tokens[:8])
print("4. Regexp Tokens (words/$$):", regex_tokens[:8])


Original Text: Dr. Smith isn't attending NLP-2026 conference! He paid $450.50 at https://nlpconf.org.
---------------------------------------------------------------------------
1. Whitespace Split Tokens : ['Dr.', 'Smith', "isn't", 'attending', 'NLP-2026', 'conference!', 'He', 'paid']
2. Treebank (word_tokenize): ['Dr.', 'Smith', 'is', "n't", 'attending', 'NLP-2026', 'conference', '!']
3. WordPunct Tokens        : ['Dr', '.', 'Smith', 'isn', "'", 't', 'attending', 'NLP']
4. Regexp Tokens (words/$$): ['Dr', 'Smith', 'isn', 't', 'attending', 'NLP', '2026', 'conference']


### 2.1 Subword Tokenization Concept
In modern NLP (like BERT, GPT), **Subword Tokenization** breaks rare, compound, or mispelled words into frequent subword units. Below is a simple character n-gram / subword segmenter showing how Out-Of-Vocabulary (OOV) words are handled:


In [3]:
def subword_ngrams(word, n=3):
    """Decomposes an unfamiliar word into subword character n-grams."""
    padded = f"<{word}>"
    return [padded[i:i+n] for i in range(len(padded) - n + 1)]

rare_words = ["unpreprocessed", "misinformation", "covid19"]
print("Subword n-gram representations for Out-Of-Vocabulary words:")
for w in rare_words:
    print(f"  {w:16} -> {subword_ngrams(w, n=3)}")


Subword n-gram representations for Out-Of-Vocabulary words:
  unpreprocessed   -> ['<un', 'unp', 'npr', 'pre', 'rep', 'epr', 'pro', 'roc', 'oce', 'ces', 'ess', 'sse', 'sed', 'ed>']
  misinformation   -> ['<mi', 'mis', 'isi', 'sin', 'inf', 'nfo', 'for', 'orm', 'rma', 'mat', 'ati', 'tio', 'ion', 'on>']
  covid19          -> ['<co', 'cov', 'ovi', 'vid', 'id1', 'd19', '19>']


## 3. Stop-Word Removal

Stop words are high-frequency words (e.g., *the, is, at, which*) that carry minimal distinctive semantic meaning for many retrieval or classification tasks.
- **NLTK Stopwords**: Contains 179 standard English words.
- **spaCy Stopwords**: Contains 326 words (includes more pronouns, auxiliaries).
- **Domain-Specific**: For social media, words like *rt, dm, via, follow* should often be pruned.


In [4]:
from nltk.corpus import stopwords

nltk_stops = set(stopwords.words('english'))
spacy_stops = nlp.Defaults.stop_words

print(f"Total NLTK English stopwords : {len(nltk_stops)}")
print(f"Total spaCy English stopwords: {len(spacy_stops)}")

# Sample noisy sentence
sentence = "This is a great tutorial and I would love to learn more about the deep learning models!"

# Filter with NLTK
tokens = word_tokenize(sentence.lower())
filtered_nltk = [t for t in tokens if t.isalnum() and t not in nltk_stops]

# Filter with spaCy
doc = nlp(sentence)
filtered_spacy = [token.text for token in doc if not token.is_stop and not token.is_punct]

print("\nOriginal Sentence:", sentence)
print("After NLTK Stopword Removal :", filtered_nltk)
print("After spaCy Stopword Removal:", filtered_spacy)


Total NLTK English stopwords : 198
Total spaCy English stopwords: 326

Original Sentence: This is a great tutorial and I would love to learn more about the deep learning models!
After NLTK Stopword Removal : ['great', 'tutorial', 'would', 'love', 'learn', 'deep', 'learning', 'models']
After spaCy Stopword Removal: ['great', 'tutorial', 'love', 'learn', 'deep', 'learning', 'models']


## 4. Stemming: PorterStemmer vs SnowballStemmer

**Stemming** is a heuristic, rule-based process that chops off suffixes (e.g., `-ed`, `-ing`, `-s`) to reach a common root form (the "stem"). The stem is not necessarily a valid dictionary word.

- **PorterStemmer (1980)**: Employs 5 sequential heuristic phases. Can be conservative or produce irregular stems.
- **SnowballStemmer (Porter 2)**: An enhanced, faster algorithmic stemmer that handles edge cases better, supports multiple languages, and reduces over-stemming.


In [5]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
snowball = SnowballStemmer(language='english')

test_words = [
    "generously", "fairly", "troubled", "caresses", 
    "ponies", "cats", "programming", "agreement", "stabilize"
]

stem_comparison = []
for word in test_words:
    p_stem = porter.stem(word)
    s_stem = snowball.stem(word)
    stem_comparison.append({"Original Word": word, "PorterStemmer": p_stem, "SnowballStemmer": s_stem})

df_stem = pd.DataFrame(stem_comparison)
df_stem


,Original Word,PorterStemmer,SnowballStemmer
0,generously,gener,generous
1,fairly,fairli,fair
2,troubled,troubl,troubl
3,caresses,caress,caress
4,ponies,poni,poni
5,cats,cat,cat
6,programming,program,program
7,agreement,agreement,agreement
8,stabilize,stabil,stabil


### Note on Stemming Errors:
1. **Over-stemming**: When two semantically distinct words are reduced to the same root (e.g., *universal*, *university*, *universe* -> *univers*).
2. **Under-stemming**: When words with the same semantic root are not reduced to the same stem (e.g., *alumnus*, *alumni*, *alumnae*).


## 5. Lemmatization: WordNet vs spaCy

**Lemmatization** uses a morphological vocabulary and morphological analysis to return the dictionary base form (the **lemma**).
- Unlike stemming, the lemma is always a valid dictionary word.
- **WordNetLemmatizer (NLTK)**: Defaults to assuming words are NOUNs unless the Part-Of-Speech (POS) is explicitly passed.
- **spaCy Lemmatizer**: Context-aware by default; uses statistical POS tags to determine the exact lemma.


In [6]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

words = ["running", "stripes", "better", "went", "corpora"]

# 1. NLTK Default (assumes Noun)
nltk_default = [lemmatizer.lemmatize(w) for w in words]

# 2. NLTK POS-Informed (passing verb/adjective)
nltk_pos_informed = [
    lemmatizer.lemmatize("running", pos=wordnet.VERB),
    lemmatizer.lemmatize("stripes", pos=wordnet.NOUN),
    lemmatizer.lemmatize("better", pos=wordnet.ADJ),
    lemmatizer.lemmatize("went", pos=wordnet.VERB),
    lemmatizer.lemmatize("corpora", pos=wordnet.NOUN)
]

# 3. spaCy Context-Aware Lemmatization
doc = nlp("He was running with stripes on his better shoes and went to study corpora.")
spacy_lemmas = [token.lemma_ for token in doc if token.text in words]

df_lemma = pd.DataFrame({
    "Word": words,
    "NLTK Default (Noun)": nltk_default,
    "NLTK (POS-Informed)": nltk_pos_informed,
    "spaCy (Context-Aware)": spacy_lemmas
})
df_lemma


,Word,NLTK Default (Noun),NLTK (POS-Informed),spaCy (Context-Aware)
0,running,running,run,run
1,stripes,stripe,stripe,stripe
2,better,better,good,well
3,went,went,go,go
4,corpora,corpus,corpus,corpora


## 6. Part-of-Speech (POS) Tagging

Part-of-Speech tagging assigns grammatical categories (Noun, Verb, Adjective, etc.) to each token in a sentence.
- **NLTK `pos_tag`**: Uses the Penn Treebank tagset (e.g., `NN`, `VBZ`, `JJ`, `NNS`).
- **spaCy**: Provides both coarse-grained Universal Dependencies POS (`token.pos_`) and fine-grained Penn Treebank tags (`token.tag_`), along with syntactic dependency parsing.


In [7]:
sample_pos_sentence = "Google AI released an exceptional NLP model yesterday."

# NLTK POS Tagging
nltk_tokens = word_tokenize(sample_pos_sentence)
nltk_tags = nltk.pos_tag(nltk_tokens)

# spaCy POS Tagging
spacy_doc = nlp(sample_pos_sentence)
spacy_tags = [(t.text, t.pos_, t.tag_, spacy.explain(t.tag_)) for t in spacy_doc]

print("=== NLTK POS Tagging (Penn Treebank) ===")
for word, tag in nltk_tags:
    print(f"  {word:15} : {tag}")

print("\n=== spaCy POS Tagging (Universal & Detailed) ===")
for text, upos, tag, explanation in spacy_tags:
    print(f"  {text:15} : {upos:6} | {tag:5} ({explanation})")


=== NLTK POS Tagging (Penn Treebank) ===
  Google          : NNP
  AI              : NNP
  released        : VBD
  an              : DT
  exceptional     : JJ
  NLP             : NNP
  model           : NN
  yesterday       : NN
  .               : .

=== spaCy POS Tagging (Universal & Detailed) ===
  Google          : PROPN  | NNP   (noun, proper singular)
  AI              : PROPN  | NNP   (noun, proper singular)
  released        : VERB   | VBD   (verb, past tense)
  an              : DET    | DT    (determiner)
  exceptional     : ADJ    | JJ    (adjective (English), other noun-modifier (Chinese))
  NLP             : PROPN  | NNP   (noun, proper singular)
  model           : NOUN   | NN    (noun, singular or mass)
  yesterday       : NOUN   | NN    (noun, singular or mass)
  .               : PUNCT  | .     (punctuation mark, sentence closer)


## 7. Comprehensive Side-by-Side Comparison: NLTK vs spaCy

Let us benchmark both frameworks on a raw paragraph across multiple preprocessing metrics:


In [8]:
raw_corpus = (
    "Natural Language Processing (NLP) enables machines to understand human language! "
    "Dr. Watson hasn't seen the latest results from AI labs, but he is analyzing the data now."
)

# Process with spaCy
doc_sp = nlp(raw_corpus)

# Process with NLTK
tokens_nl = word_tokenize(raw_corpus)
pos_nl = dict(nltk.pos_tag(tokens_nl))

comparison_data = []
for token in doc_sp:
    if token.text.isalnum():
        sp_lemma = token.lemma_
        sp_pos = token.pos_
        nl_pos = pos_nl.get(token.text, 'N/A')
        nl_stem = snowball.stem(token.text)
        nl_lemma = lemmatizer.lemmatize(token.text.lower())
        
        comparison_data.append({
            "Token": token.text,
            "NLTK Stem": nl_stem,
            "NLTK Lemma": nl_lemma,
            "NLTK POS": nl_pos,
            "spaCy Lemma": sp_lemma,
            "spaCy Universal POS": sp_pos
        })

df_comp = pd.DataFrame(comparison_data)
df_comp.head(12)


,Token,NLTK Stem,NLTK Lemma,NLTK POS,spaCy Lemma,spaCy Universal POS
0,Natural,natur,natural,JJ,Natural,PROPN
1,Language,languag,language,NNP,Language,PROPN
2,Processing,process,processing,NNP,Processing,PROPN
3,NLP,nlp,nlp,NNP,NLP,PROPN
4,enables,enabl,enables,VBZ,enable,VERB
5,machines,machin,machine,NNS,machine,NOUN
6,to,to,to,TO,to,PART
7,understand,understand,understand,VB,understand,VERB
8,human,human,human,JJ,human,ADJ
9,language,languag,language,NN,language,NOUN


### Architectural Summary: NLTK vs spaCy

| Feature | NLTK | spaCy |
| :--- | :--- | :--- |
| **Philosophy** | Academic, experimental, algorithmic toolkit | Production-ready, industrial strength, opinionated |
| **Speed** | Python-based, slower on large corpora | Cython-optimized, extremely fast |
| **Lemmatization** | WordNet lookup (requires manual POS mapping) | Integrated neural/statistical pipeline |
| **POS Tagging** | Perceptron tagger (Penn Treebank) | Universal Dependencies + Penn Treebank |
| **Extensibility** | Huge collection of algorithms & corpora | Custom pipeline components, transformer models |


## 8. Real-World Exemplar: Preprocessing Multilingual & Social Media Text for Fake-News Detection

### Problem Statement:
Social media posts scraped from Twitter/Reddit frequently contain noise:
- Hyperlinks (`https://...`)
- User mentions (`@username`)
- Hashtags (`#breakingnews`)
- HTML entities (`&amp;`, `&lt;`)
- Emojis (`🚨`, `🔥`, `🛑`)
- Code-mixed words (e.g., Hinglish: *"Yeh news bilkul fake hai"*)

We build a modular, reusable **`SocialMediaPreprocessor`** pipeline that normalizes this noisy text into clean tokens ready for downstream fake-news classifiers.


In [9]:
class SocialMediaPreprocessor:
    """
    A reusable preprocessing pipeline for noisy multilingual social media
    and news corpora, tailored for fake-news detection systems.
    """
    def __init__(self, engine='spacy', remove_stopwords=True, lemmatize=True):
        self.engine = engine.lower()
        self.remove_stopwords = remove_stopwords
        self.lemmatize = lemmatize
        self.nltk_stopwords = set(stopwords.words('english'))
        # Add common social media noise words to stopwords
        self.nltk_stopwords.update({'rt', 'via', 'amp', 'dm', 'fake', 'news'})
        self.lemmatizer = WordNetLemmatizer()
        # Reference global nlp pipeline initialized earlier
        global nlp
        self.nlp = nlp
        
    def clean_raw_text(self, text):
        """Applies regex cleaning for URLs, mentions, emojis, and symbols."""
        # Convert HTML entities
        text = re.sub(r'&amp;', '&', text)
        text = re.sub(r'&lt;|&gt;', ' ', text)
        # Remove URLs
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        # Remove User Mentions (@user)
        text = re.sub(r'@\w+', '', text)
        # Demojize: Convert emojis to descriptive text (e.g. 🚨 -> :warning:)
        text = emoji.demojize(text, delimiters=(" ", " "))
        # Replace hashtags with just the word: #FakeNews -> FakeNews
        text = re.sub(r'#(\w+)', r'\1', text)
        # Remove special characters, numbers, and extra symbols
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        return text.lower()

    def process(self, text):
        """Full pipeline execution based on selected engine."""
        clean_text = self.clean_raw_text(text)
        
        if self.engine == 'spacy':
            doc = self.nlp(clean_text)
            processed_tokens = []
            for token in doc:
                if self.remove_stopwords and (token.is_stop or token.text in self.nltk_stopwords):
                    continue
                if len(token.text) <= 2:
                    continue
                lemma = token.lemma_ if self.lemmatize else token.text
                processed_tokens.append(lemma)
            return " ".join(processed_tokens)
            
        else: # NLTK engine
            tokens = word_tokenize(clean_text)
            processed_tokens = []
            for token in tokens:
                if self.remove_stopwords and token in self.nltk_stopwords:
                    continue
                if len(token) <= 2:
                    continue
                lemma = self.lemmatizer.lemmatize(token) if self.lemmatize else token
                processed_tokens.append(lemma)
            return " ".join(processed_tokens)


### Testing the Pipeline on a Benchmark Dataset of Scraped Posts
Let's construct a sample dataset containing realistic fake-news and real-news posts with emojis, code-mixing, and noisy tags:


In [10]:
# Sample dataset simulating scraped Twitter/Reddit posts
raw_social_posts = [
    "🚨 BREAKING: 5G radiation causes instant virus mutation! Read here 👉 https://fake-wire.com/alert #COVID19 #5GExposed @WorldAlert",
    "Scientists confirm breakthrough in mRNA cancer vaccines after successful phase 3 trials. Details: https://nature.com/articles/med98",
    "Sach me shocking news! 😱 Secret meeting exposes alien technology hidden in Antarctica!! #AlienProof @conspiracy_hub",
    "Government announces new subsidised solar energy scheme for rural households starting next month.",
    "RT @crypto_guru: Guaranteed 1000% ROI in 24 hours!! Send 1 ETH to get 5 ETH back 🚀💰🔥 Visit: http://scam-crypto.io",
    "WHO releases updated guidelines on daily physical activity and mental health well-being."
]

labels = ["Fake", "Real", "Fake", "Real", "Fake", "Real"]

# Instantiate pipeline using both engines
spacy_pipeline = SocialMediaPreprocessor(engine='spacy', remove_stopwords=True, lemmatize=True)
nltk_pipeline = SocialMediaPreprocessor(engine='nltk', remove_stopwords=True, lemmatize=True)

cleaned_spacy = [spacy_pipeline.process(post) for post in raw_social_posts]
cleaned_nltk = [nltk_pipeline.process(post) for post in raw_social_posts]

df_benchmark = pd.DataFrame({
    "Original Post": raw_social_posts,
    "Label": labels,
    "Cleaned (spaCy Pipeline)": cleaned_spacy,
    "Cleaned (NLTK Pipeline)": cleaned_nltk
})

pd.set_option('display.max_colwidth', None)
df_benchmark


,Original Post,Label,Cleaned (spaCy Pipeline),Cleaned (NLTK Pipeline)
0,🚨 BREAKING: 5G radiation causes instant virus mutation! Read here 👉 https://fake-wire.com/alert #COVID19 #5GExposed @WorldAlert,Fake,police car light break radiation cause instant virus mutation read backhand index pointing right covid gexpose,police car light breaking radiation cause instant virus mutation read backhand index pointing right covid gexposed
1,Scientists confirm breakthrough in mRNA cancer vaccines after successful phase 3 trials. Details: https://nature.com/articles/med98,Real,scientist confirm breakthrough mrna cancer vaccine successful phase trial detail,scientist confirm breakthrough mrna cancer vaccine successful phase trial detail
2,Sach me shocking news! 😱 Secret meeting exposes alien technology hidden in Antarctica!! #AlienProof @conspiracy_hub,Fake,sach shocking face scream fear secret meeting expose alien technology hide antarctica alienproof,sach shocking face screaming fear secret meeting expose alien technology hidden antarctica alienproof
3,Government announces new subsidised solar energy scheme for rural households starting next month.,Real,government announce new subsidise solar energy scheme rural household start month,government announces new subsidised solar energy scheme rural household starting next month
4,RT @crypto_guru: Guaranteed 1000% ROI in 24 hours!! Send 1 ETH to get 5 ETH back 🚀💰🔥 Visit: http://scam-crypto.io,Fake,guarantee roi hour send eth eth rocket money bag fire visit,guaranteed roi hour send eth get eth back rocket money bag fire visit
5,WHO releases updated guidelines on daily physical activity and mental health well-being.,Real,release update guideline daily physical activity mental health,release updated guideline daily physical activity mental health well


## 9. Conclusion & Key Takeaways

1. **Tokenization Nuances:** Rule-based tokenizers like NLTK's `word_tokenize` and spaCy correctly split contractions (`isn't` -> `is` + `n't`), which is critical for sentiment and intent classification.
2. **Stemming vs Lemmatization:** Stemming is fast but crude (often yielding non-words like `poni` or `stabil`), whereas Lemmatization preserves semantic validity (`pony`, `stabilize`).
3. **NLTK vs spaCy in Production:** 
   - spaCy is significantly faster and natively handles POS tagging alongside lemmatization in a unified pipeline.
   - NLTK is highly modular for academic research and custom linguistic rules.
4. **Social Media Text Normalization:** Removing URLs, user mentions, and normalizing emojis while preserving core contextual lemmas yields clean, high-signal tokens ideal for downstream fake-news classifiers.
